# Semaine 2 — Jour 6 — Agent Loop & planification — Notebook formateur

Ce notebook inclut les éléments de correction et les notes de review.

## Démonstration de référence

In [ ]:
from pathlib import Path
import sys

candidate_paths = [
    Path("../../book/week02/day06/labs").resolve(),
    Path("../book/week02/day06/labs").resolve(),
    Path("book/week02/day06/labs").resolve(),
]
for candidate in candidate_paths:
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break

from agent_loop_planner import AgentLoop, build_default_registry

In [ ]:
loop = AgentLoop(build_default_registry(), max_iterations=5)
state = loop.run("Bonjour, ma commande A-100 est en retard. Pouvez-vous m'aider ?")
print(state.to_json())

## Corrigé synthétique

Une bonne boucle contient : état explicite, policy, registre d'outils, reducer, condition d'arrêt et trace.

# Corrigé — Exercices — Jour 6

## Corrigé exercice 1

1. `lookup_order({"order_id": "A-100"})` : action.
2. `status=delayed, eta=2026-09-02` : observation.
3. `Préparer une réponse pour un client dont la commande est en retard` : goal.
4. `order_id=A-100, missing_facts=[]` : state.
5. `Récupérer l'état de la commande puis rédiger une réponse` : plan.
6. `Bonjour, votre commande est retardée et devrait arriver le 2 septembre.` : final answer.

## Corrigé exercice 2

Problèmes :

- boucle infinie possible ;
- absence de limite d'itérations ;
- absence de condition d'arrêt ;
- absence de gestion d'erreur ;
- absence de journalisation ;
- action non validée ;
- outil inconnu possible ;
- aucune distinction entre observation et final answer.

Pseudo-code amélioré :

```python
state = initialize_state(goal)
max_iterations = 5

while not state.done and state.iterations < max_iterations:
    action = choose_action(state)

    if not registry.has_tool(action.tool_name):
        state.status = "error"
        break

    observation = registry.execute(action)
    state = reducer.apply(state, action, observation)

    if should_finish(state):
        state.status = "completed"
        state.done = True

if not state.done and state.iterations >= max_iterations:
    state.status = "max_iterations_reached"
```

## Corrigé exercice 3

Le plan ne peut pas continuer car l'étape `lookup_order` nécessite un `order_id`.

Le message utilisateur ne contient aucun identifiant exploitable. L'agent ne doit pas inventer de valeur.

Replanification :

```text
1. Demander à l'utilisateur son identifiant de commande.
2. Mettre le statut à waiting_for_user.
3. Suspendre la boucle.
```

## Corrigé exercice 4

Exemple :

```json
{
  "iteration": 1,
  "action": "lookup_order",
  "arguments": {
    "order_id": "A-100"
  },
  "success": true,
  "updated_fields": ["order_status", "eta"]
}
```

## Corrigé exercice 5

Rôles :

- `AgentState` : état courant de résolution.
- `PlanStep` : étape de planification.
- `ToolAction` : action concrète à exécuter.
- `Observation` : résultat d'une action.
- `ToolRegistry` : registre contrôlé des outils autorisés.
- `AgentLoop` : orchestration de la boucle complète.

## Corrigé exercice 6

Le système proposé n'est pas fiable car il délègue toute l'architecture au prompt.

Il manque :

- un état explicite ;
- une liste d'outils autorisés ;
- une validation des arguments ;
- une condition d'arrêt ;
- une trace d'exécution ;
- une gestion d'erreur ;
- une limite d'itérations.

Architecture proposée :

```text
Planner -> Policy -> ToolRegistry -> Reducer
```

- `Planner` produit le plan.
- `Policy` choisit l'action suivante.
- `ToolRegistry` exécute uniquement les outils autorisés.
- `Reducer` met à jour l'état avec les observations.

# Corrigé — Questions d'entretien — Jour 6

## Réponse 1

Un chatbot simple transforme un message en réponse.

Un agent avec boucle agentique peut planifier, appeler des outils, observer des résultats, mettre à jour un état et décider de continuer ou de terminer.

## Réponse 2

La séparation action/observation rend le système traçable et testable.

L'action décrit ce que l'agent veut faire. L'observation décrit ce qui s'est réellement passé.

## Réponse 3

Une condition d'arrêt est une règle qui indique quand la boucle doit se terminer.

Exemples :

- objectif atteint ;
- réponse finale produite ;
- information utilisateur manquante ;
- erreur non récupérable ;
- limite d'itérations atteinte.

## Réponse 4

On peut éviter les appels répétitifs en combinant :

- limite d'itérations ;
- journal des actions déjà exécutées ;
- détection d'action identique ;
- état mis à jour après chaque observation ;
- politique empêchant un appel sans nouvelle information.

## Réponse 5

La planification initiale produit une stratégie avant l'exécution.

La replanification adapte cette stratégie après une observation, une erreur ou une nouvelle contrainte.

## Réponse 6

Il faut demander une clarification lorsque l'information manquante bloque une action sûre.

Exemple : l'agent ne peut pas appeler `lookup_order` sans identifiant de commande.

## Réponse 7

On peut remplacer le LLM par une policy déterministe ou un simulateur.

Cela permet de tester :

- les transitions d'état ;
- la sélection d'actions ;
- les erreurs ;
- les limites d'itérations ;
- les statuts finaux.

## Réponse 8

Le state représente la connaissance courante de résolution.

Il relie le goal, le plan, les observations, les champs collectés, les actions passées et le statut final.

## Réponse 9

La traçabilité permet de comprendre pourquoi l'agent a agi.

Elle est essentielle pour :

- déboguer ;
- auditer ;
- évaluer ;
- améliorer les prompts ;
- analyser les coûts ;
- sécuriser la production.

## Réponse 10

Le function calling structure les actions possibles.

Les Structured Outputs structurent les décisions, plans et réponses.

Ensemble, ils réduisent l'ambiguïté entre le modèle et l'orchestrateur applicatif.

# Corrigé — Challenge — Jour 6

## Solution attendue

Une solution correcte doit contenir :

- un état explicite ;
- un extracteur d'identifiant de commande ;
- un outil `lookup_order`;
- un outil `draft_support_reply`;
- une boucle bornée ;
- un journal d'exécution ;
- une condition d'arrêt ;
- un statut `completed` ou `waiting_for_user`.

## Exemple de déroulement nominal

Entrée :

```text
Bonjour, ma commande A-100 est en retard. Pouvez-vous m'aider ?
```

État initial :

```json
{
  "goal": "Résoudre une demande support",
  "order_id": "A-100",
  "order_status": null,
  "eta": null,
  "final_answer": null,
  "status": "running"
}
```

Itération 1 :

```json
{
  "action": "lookup_order",
  "arguments": {"order_id": "A-100"},
  "observation": {"status": "delayed", "eta": "2026-09-02"}
}
```

Itération 2 :

```json
{
  "action": "draft_support_reply",
  "arguments": {
    "order_id": "A-100",
    "status": "delayed",
    "eta": "2026-09-02"
  }
}
```

État final :

```json
{
  "status": "completed",
  "final_answer": "Bonjour, votre commande A-100 est actuellement retardée et sa livraison est estimée au 2026-09-02."
}
```

## Variante avec clarification

Entrée :

```text
Bonjour, ma commande est en retard.
```

L'agent doit produire :

```json
{
  "status": "waiting_for_user",
  "final_answer": "Pouvez-vous me transmettre votre identifiant de commande ?"
}
```

## Points de vigilance

Une solution incorrecte :

- invente un identifiant de commande ;
- appelle `lookup_order` sans argument ;
- ignore les erreurs d'outil ;
- ne journalise rien ;
- ne possède pas de limite d'itérations ;
- confond mémoire longue durée et state courant.

# Review formateur — Jour 6

## Résumé pédagogique

Cette journée introduit la compétence centrale du développement agentique : construire une boucle contrôlée autour du modèle.

L'apprenant doit comprendre qu'un agent fiable n'est pas un prompt autonome, mais une orchestration logicielle.

## Points à vérifier

Le formateur doit vérifier que l'apprenant sait expliquer :

- ce qu'est une boucle agentique ;
- pourquoi une limite d'itérations est obligatoire ;
- pourquoi l'état doit être explicite ;
- comment une observation modifie le state ;
- quand demander une clarification ;
- comment tester une boucle sans LLM réel.

## Questions de relance

- Que se passe-t-il si l'outil retourne une erreur ?
- Quelle différence entre un plan et une action ?
- Pourquoi faut-il journaliser les itérations ?
- Comment empêcher un agent de répéter la même action ?
- Quelle information doit rester dans le state et quelle information doit aller en mémoire longue durée ?

## Erreurs fréquentes à surveiller

1. L'apprenant confond planification et exécution.
2. L'apprenant met toute la logique dans le prompt.
3. L'apprenant oublie la condition d'arrêt.
4. L'apprenant ne valide pas les outils.
5. L'apprenant ne distingue pas action et observation.
6. L'apprenant invente des données manquantes.

## Évaluation rapide

L'apprenant est prêt pour le Jour 7 s'il peut concevoir une boucle qui :

- extrait ou demande les informations nécessaires ;
- appelle les bons outils ;
- observe les résultats ;
- met à jour l'état ;
- termine avec un statut clair ;
- produit une trace lisible.

## Propositions d'amélioration

Aucune modification de spécification n'est proposée.

Une amélioration pédagogique possible, sans changer la roadmap, serait d'ajouter plus tard un exercice comparatif entre :

- boucle contrôlée côté application ;
- boucle gérée par un framework agentique ;
- boucle multi-agent avec délégation.